# AI-Powered Skin Cancer Detection using Custom CNN with Explainable AI (Grad-CAM)

## Cell 1: Imports

In [ ]:
import os
import time
import pickle
import itertools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('Modules loaded successfully.')

## Cell 2: Kaggle Dataset Setup

In [ ]:
from google.colab import files
print('Upload kaggle.json (Kaggle API credentials):')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d ./dataset/

## Cell 3: Metadata Loading

In [ ]:
metadata_path = './dataset/HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)
print('Metadata shape:', df.shape)
df.head()

## Cell 4: Image Path Creation

In [ ]:
image_dir_part1 = './dataset/HAM10000_images_part_1'
image_dir_part2 = './dataset/HAM10000_images_part_2'

def get_image_path(image_id):
    path1 = os.path.join(image_dir_part1, image_id + '.jpg')
    if os.path.exists(path1):
        return path1
    path2 = os.path.join(image_dir_part2, image_id + '.jpg')
    return path2

df['image_path'] = df['image_id'].apply(get_image_path)
print('Image paths created.')

## Cell 5: Label Encoding

In [ ]:
le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['dx'])
print('Class mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

## Cell 6: Train-Validation-Test Split

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['encoded_label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['encoded_label'], random_state=42)

print(f'Train set size: {len(train_df)}')
print(f'Validation set size: {len(val_df)}')
print(f'Test set size: {len(test_df)}')

## Cell 7: Image Generators

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    rescale=1./255
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='dx',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='dx',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_path',
    y_col='dx',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

## Cell 8: Class Weights

In [ ]:
class_counts = train_df['dx'].value_counts().to_dict()
total_samples = len(train_df)
num_classes = len(class_counts)

class_weights_dict = {}
for class_name, count in class_counts.items():
    class_idx = list(train_generator.class_indices.keys()).index(class_name)
    class_weights_dict[class_idx] = total_samples / (num_classes * count)

print('Class weights:', class_weights_dict)

## Cell 9: Custom CNN Architecture

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(7, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Cell 10: Training

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.2, patience=3, monitor='val_loss'),
    ModelCheckpoint('skin_cancer_model_best.keras', save_best_only=True, monitor='val_loss')
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    epochs=40,
    callbacks=callbacks
)

## Cell 11: Evaluation

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Graph')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Graph')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')

y_true = test_generator.classes
y_pred_probs = model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
print(f'\nWeighted Precision: {precision:.4f}')
print(f'Weighted Recall: {recall:.4f}')
print(f'Weighted F1-Score: {f1:.4f}')

## Cell 12: Save Model

In [ ]:
model.save('skin_cancer_model.h5')
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('Model and label encoder saved.')

## Cell 13: Single Image Prediction

In [ ]:
def predict_single_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE)
    img_array = img / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    predictions = model.predict(img_array, verbose=0)[0]
    predicted_class_idx = np.argmax(predictions)
    predicted_class = le.inverse_transform([predicted_class_idx])[0]
    confidence = predictions[predicted_class_idx]
    
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Predicted: {predicted_class}\nConfidence: {confidence:.4f}')
    plt.show()
    
    print('Class probabilities:')
    for class_name, prob in zip(le.classes_, predictions):
        print(f'{class_name}: {prob:.4f}')

# Example usage with a test image
sample_image_path = test_df['image_path'].iloc[0]
predict_single_image(sample_image_path)

## Cell 14: Grad-CAM

In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, Conv2D):
            return layer.name
    return None

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(image_path, model, last_conv_layer_name, alpha=0.4):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, IMAGE_SIZE)
    img_array = img_resized / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * alpha + img
    superimposed_img = np.uint8(superimposed_img)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap)
    plt.title('Grad-CAM Heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(superimposed_img)
    plt.title('Overlay')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

last_conv_layer = get_last_conv_layer(model)
print('Last Conv2D layer:', last_conv_layer)

sample_image_path = test_df['image_path'].iloc[0]
display_gradcam(sample_image_path, model, last_conv_layer)